# Text-to-SQL Agent — Testing Notebook

Tests the **DuckDB text-to-SQL agent** (`src/agents/sql_analyst.py`), built as an
explicit LangGraph pipeline:

```
generate  ->  execute  ->  respond
(reasoning     (DuckDB      (final
 + SQL)         runs it)     answer)
```

On a SQL error the graph loops back to `generate` so the agent self-corrects. Every
number is computed **in SQL** — the model never does arithmetic itself. `ask_sql`
returns a structured result: **reasoning, sql, answer, data**.

Two parts: (1) the **DuckDB layer + safety guard** — no API calls; (2) the **agent**.

## Setup

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

## The graph

The compiled LangGraph, as a mermaid diagram.

In [2]:
from src.agents.sql_analyst import sql_agent

print(sql_agent.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	generate(generate)
	execute(execute)
	respond(respond)
	__end__([<p>__end__</p>]):::last
	__start__ --> generate;
	execute -.-> generate;
	execute -.-> respond;
	generate -.-> execute;
	generate -.-> respond;
	respond --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## Part 1 — DuckDB layer & safety guard (no API calls)

The ledger parquet is exposed as a read-only DuckDB view called `ledger`. Only
SELECT/WITH queries run; anything else is rejected before it executes.

In [3]:
from src.agents.sql_analyst import _is_safe, _execute_sql

# The safety guard
checks = {
    "SELECT * FROM ledger": True,
    "WITH x AS (SELECT 1) SELECT * FROM x": True,
    "DROP TABLE ledger": False,
    "SELECT 1; DROP TABLE ledger": False,   # stacked statements
    "COPY ledger TO 'out.csv'": False,       # filesystem write
}
for q, expected in checks.items():
    print(f"{_is_safe(q)!s:5}  (expected {expected!s:5})  {q}")

True   (expected True )  SELECT * FROM ledger
True   (expected True )  WITH x AS (SELECT 1) SELECT * FROM x
False  (expected False)  DROP TABLE ledger
False  (expected False)  SELECT 1; DROP TABLE ledger
False  (expected False)  COPY ledger TO 'out.csv'


In [4]:
# Run SQL directly through the executor
_execute_sql("SELECT ledger_type, ROUND(SUM(profit), 2) AS pnl "
             "FROM ledger GROUP BY ledger_type")

{'data': [{'ledger_type': 'revenue', 'pnl': 2887652.89},
  {'ledger_type': 'expenses', 'pnl': -1354321.02}]}

In [5]:
# An unsafe query is rejected, not executed
_execute_sql("DELETE FROM ledger")

{'error': 'Only read-only SELECT queries are allowed.'}

In [6]:
# A broken query returns the SQL error (which the agent uses to self-correct)
_execute_sql("SELECT nope FROM ledger")

{'error': 'SQL error: Binder Error: Referenced column "nope" not found in FROM clause!\nCandidate bindings: "property_name"\n\nLINE 1: SELECT nope FROM ledger\n               ^'}

## Part 2 — The SQL agent (makes Claude calls)

`ask_sql(question)` returns `{reasoning, sql, answer, data}`. The helper below prints
the reasoning, the SQL the agent wrote, and its final answer.

In [7]:
from src.agents.sql_analyst import ask_sql


def show(question: str):
    """Run the agent and print its reasoning, SQL, and answer."""
    r = ask_sql(question)
    print("Q:", question)
    print("\nREASONING:", r["reasoning"])
    print("\nSQL:", " ".join(r["sql"].split()) if r["sql"] else "(none — not answerable)")
    print("\nANSWER:", r["answer"])
    print("=" * 80)

### Straightforward questions

In [8]:
show("What is the total P&L for all properties in 2024?")

Q: What is the total P&L for all properties in 2024?

REASONING: The ledger table contains a profit column that represents net P&L (revenue positive, expenses negative). I can sum all profit entries where year = '2024' to get the total P&L for all properties in 2024.

SQL: SELECT SUM(profit) AS total_pl_2024 FROM ledger WHERE year = '2024';

ANSWER: The total P&L for all properties in 2024 is **$1,171,521.55**.


In [9]:
show("Who are my top 3 tenants by profit?")

Q: Who are my top 3 tenants by profit?

REASONING: The question asks for the top 3 tenants ranked by their total profit contribution. I can aggregate profit by tenant_name, sort in descending order, and limit to 3 results. Only tenant-specific rows (where tenant_name is NOT NULL) should be included.

SQL: SELECT tenant_name, SUM(profit) AS total_profit FROM ledger WHERE tenant_name IS NOT NULL GROUP BY tenant_name ORDER BY total_profit DESC LIMIT 3

ANSWER: Your top 3 tenants by profit are:

1. **Tenant 7** – $880,512.18
2. **Tenant 14** – $391,490.29
3. **Tenant 11** – $292,531.00


In [10]:
show("What were my biggest expense categories in 2024?")

Q: What were my biggest expense categories in 2024?

REASONING: The ledger table contains expense records with ledger_category and profit (negative for expenses). We can filter for expenses in year 2024, group by ledger_category, sum the profit (expense amounts), and order by the largest absolute values to find the biggest expense categories.

SQL: SELECT ledger_category, SUM(profit) AS total_expense FROM ledger WHERE ledger_type = 'expenses' AND year = '2024' GROUP BY ledger_category ORDER BY ABS(total_expense) DESC;

ANSWER: Your biggest expense categories in 2024 were:

1. **Interest Mortgage** – $537,260
2. **Success Fees** – $180,000
3. **Asset Management Fees** – $117,000
4. **Property Management Fees** – $96,764
5. **Real Estate Taxes** – $49,728

These five categories accounted for the vast majority of your 2024 expenses, with mortgage interest alone representing nearly 60% of total spending.


### Compound / comparison questions (all arithmetic done in SQL)

In [11]:
show("How does Q1 2025 compare to Q1 2024 in profit, and what drove the difference?")

Q: How does Q1 2025 compare to Q1 2024 in profit, and what drove the difference?

REASONING: The question asks for a comparison of Q1 2025 vs Q1 2024 profit, and what drove the difference. The ledger table contains profit data by quarter and by ledger_group, so I can aggregate total profit for each quarter and also break down by ledger_group (revenue vs expenses) to identify the drivers of the difference.

SQL: SELECT COALESCE(quarter, 'TOTAL') AS quarter, COALESCE(ledger_group, 'All Groups') AS ledger_group, SUM(profit) AS total_profit FROM ledger WHERE quarter IN ('2025-Q1', '2024-Q1') GROUP BY ROLLUP(quarter, ledger_group) ORDER BY quarter DESC, ledger_group;

ANSWER: ## Q1 2025 vs Q1 2024 Profit Comparison

**Overall Profit:**
- **Q1 2025:** $361,810.32
- **Q1 2024:** $262,309.07
- **Improvement:** $99,501.25

**Key Drivers of the Difference:**

| Ledger Group | Q1 2025 | Q1 2024 | Change |
|---|---|---|---|
| Rental Income | $627,285.70 | $589,634.62 | +$37,651.08 |
| General Expe

In [12]:
show("Who are my top tenants, and is anything unusual in the numbers?")

Q: Who are my top tenants, and is anything unusual in the numbers?

REASONING: 
The ledger table contains tenant_name and profit fields, allowing us to:
1. Identify all tenants and their total profit contribution (sum of all transactions)
2. Rank tenants by total profit to find the "top" tenants
3. Examine the composition of their revenue vs. expenses to spot anything unusual (e.g., tenants with negative total profit, or highly skewed expense patterns)

I'll aggregate profit by tenant_name, order by total profit descending to identify leaders, and also break down by ledger_type to reveal the revenue/expense split for each tenant—this makes it easy to spot anomalies like tenants with large expense items.


SQL: SELECT tenant_name, SUM(profit) FILTER (WHERE ledger_type = 'revenue') AS total_revenue, SUM(profit) FILTER (WHERE ledger_type = 'expenses') AS total_expenses, SUM(profit) AS net_profit, COUNT(*) AS transaction_count, COUNT(DISTINCT property_name) AS properties_involved FROM ledg

### Robustness — out-of-scope, unknown, and relative-time inputs

In [13]:
# Out of scope: the ledger has no valuations -> answerable=false, graceful decline
show("What is the market price of Building 17?")

Q: What is the market price of Building 17?

REASONING: The ledger table contains only financial transactions (revenue and expenses) recorded by month, quarter, and year. It does not contain asset valuations, appraisals, or market prices for properties. Market price of Building 17 would require external market data or property valuation records, which are not available in this ledger.

SQL: (none — not answerable)

ANSWER: I cannot answer this question from the available financial data. The ledger contains only transaction records (revenue and expenses) and does not include asset valuations, appraisals, or market prices for properties like Building 17. To find the market price of Building 17, you would need to consult property valuation records or external market data sources.


In [14]:
# Unknown property -> query returns nothing -> honest 'no data' answer
show("What is the P&L for Building 999?")

Q: What is the P&L for Building 999?

REASONING: The question asks for the P&L (Profit & Loss, i.e., net profit) for Building 999. This is answered by summing the profit column filtered for property_name = 'Building 999'. The profit column already contains the net result (revenue positive, expenses negative), so a simple SUM() gives the total P&L.

SQL: SELECT SUM(profit) AS p_and_l FROM ledger WHERE property_name = 'Building 999';

ANSWER: The query returned no data for Building 999. Either this building does not exist in the ledger, or it has no profit/loss records.


In [15]:
# Relative time: resolved against the latest period in the data
show("What was my profit last quarter?")

Q: What was my profit last quarter?

REASONING: The user is asking for profit in the last quarter relative to the latest data point (2025-M03, which is in 2025-Q1). "Last quarter" means the quarter before 2025-Q1, which is 2024-Q4. The ledger table contains a 'profit' column and a 'quarter' column, so I can sum profit where quarter = '2024-Q4'.

SQL: SELECT SUM(profit) AS profit_last_quarter FROM ledger WHERE quarter = '2024-Q4';

ANSWER: Your profit for Q4 2024 was **$278,954.87**.


## Notes

- **Structured output:** the agent returns `reasoning`, `sql`, `answer`, and `data`,
  so the reasoning and query are transparent — useful for the step-by-step requirement.
- **No mental math:** the LLM only writes SQL and phrases results; DuckDB does every
  calculation, including differences and percentages (conditional aggregation).
- **Self-correction:** a failed query is fed back to `generate`, which rewrites it.
- **Scope:** out-of-scope questions set `answerable=false` and are declined without SQL.
  (Detecting *vague* questions and routing will be the Router agent's job, added next.)